# Z2 index from Wilson-loop (Berry-phase) spectra

Formerly `MnBi2Te4/Z2_index.ipynb`; now takes any Wannier run. **Still on the old path**, see `notes/code.md`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

from pythtb import W90, TBModel, Mesh, WFArray
import pythtb
import logging

pythtb.configure_logging(level="DEBUG")

In [ ]:
PREFIX = "MnBi2Te4"
RUN = DATA / "MnBi2Te4/base/vdw_U/U_0.0/150300bc889"   # any Wannier run
N_OCC = 58                                              # occupied Wannier bands
w90 = W90(str(RUN), PREFIX)

In [ ]:
model = w90.model(
    zero_energy=0, min_hopping_norm=1e-5,
    max_distance=125, ignorable_imaginary_part=None
    )

In [ ]:
mesh = Mesh(['k','k','k'])
nks = (8, 8, 8)  # use at least 8^3 for stable Z2 parity
mesh.build_grid(nks)
print(mesh)

In [ ]:
wfa = WFArray(model.lattice, mesh)

In [ ]:
wfa.solve_model(model, use_tensorflow=True)

In [ ]:
n_occ = N_OCC

In [ ]:
b_phase_kx = wfa.berry_phase(0, np.arange(n_occ), berry_evals=True)
b_phase_ky = wfa.berry_phase(1, np.arange(n_occ), berry_evals=True)
b_phase_kz = wfa.berry_phase(2, np.arange(n_occ), berry_evals=True)

In [ ]:
b_phase_kx.shape

In [ ]:
import numpy as np

TWOPI = 2*np.pi

def wrap_0_2pi(theta):
    """Map angles to [0, 2pi)."""
    return np.mod(theta, TWOPI)

def largest_gap_center(thetas):
    """Return midpoint of the largest cyclic gap in sorted angles."""
    th = np.sort(wrap_0_2pi(thetas))
    gaps = np.diff(np.r_[th, th[0] + TWOPI])
    j = np.argmax(gaps)
    return (th[j] + 0.5 * gaps[j]) % TWOPI

def unwrap_on_circle(angles):
    """Continuous unwrap for angles on [0, 2pi)."""
    ang = np.asarray(angles, dtype=float)
    out = np.zeros_like(ang)
    out[0] = ang[0]
    for i in range(1, len(ang)):
        d = (ang[i] - ang[i-1] + np.pi) % TWOPI - np.pi
        out[i] = out[i-1] + d
    return out

def z2_from_wcc_flow(wcc_phases):
    """
    Compute 2D Z2 from Wilson-loop phases theta_n(ky) on ky in [0, pi].
    Robust formulation: use winding of the center of the largest WCC gap.
    Returns (nu, winding_integer, unwrapped_gap_centers).
    """
    wcc = wrap_0_2pi(wcc_phases)
    gap_centers = np.array([largest_gap_center(row) for row in wcc])
    gap_unwrapped = unwrap_on_circle(gap_centers)
    winding = int(np.rint((gap_unwrapped[-1] - gap_unwrapped[0]) / np.pi))
    nu = winding % 2
    return nu, winding, gap_unwrapped

def z2_on_plane_from_bphase_kx(b_phase_kx, kz_index, ky_half=True):
    """
    Compute nu on plane kz = kz_index using Wilson loops along kx vs ky.
    b_phase_kx shape: (Nky, Nkz, Nocc).
    If ky_half=True, use ky in [0, pi] (first half of uniform 0..2pi mesh).
    """
    wcc = b_phase_kx[:, kz_index, :]  # (Nky, Nocc)
    if ky_half:
        Nky = wcc.shape[0]
        wcc = wcc[:(Nky // 2 + 1), :]
    return z2_from_wcc_flow(wcc)

In [ ]:
# Inputs you already have
# b_phase_kx shape should be (Nky, Nkz, Nocc)

Nky, Nkz, Nocc = b_phase_kx.shape

kz0 = 0
kzpi = Nkz // 2   # works if kz grid is uniform 0..2π with Nkz even

nu_kz0, wind_kz0, gap_kz0 = z2_on_plane_from_bphase_kx(b_phase_kx, kz0, ky_half=True)
nu_kzpi, wind_kzpi, gap_kzpi = z2_on_plane_from_bphase_kx(b_phase_kx, kzpi, ky_half=True)

# Strong index ν0 = ν(kz=0) XOR ν(kz=π)
nu0 = (nu_kz0 ^ nu_kzpi)

print("Plane Z2 (largest-gap winding):")
print(f"  nu(kz=0)  = {nu_kz0}   (gap winding={wind_kz0} * pi)")
print(f"  nu(kz=pi) = {nu_kzpi}  (gap winding={wind_kzpi} * pi)")
print(f"Strong nu0 = {nu0}")